# 05 — Feature Engineering & Data Split

## Goal

This notebook exists to **test two things**:

1. **The created features** — that every column produced by `src/build_features.py` is what it claims to be. Correct grain, correct derivation from the source columns, no missing values, no impossible values, and no column that silently encodes something other than its name.
2. **The train / validation / test split** — that the split is reproducible, that it partitions the customers cleanly, and that the three parts are comparable enough for a model tuned on one to be evaluated fairly on another.

Neither is exploratory. `04_eda_first_txn.ipynb` asked *what does the data say*; this notebook asks *is what we built correct*. Every cell here should be a check with a pass/fail reading, not a chart to interpret.

**Source:** `data/processed/customer_features.csv` — the modelling table, one row per customer, written by `src/build_features.py`.

## What is being tested

### The features

`build_features.py` collapses 125,042 corrected line items into 5,044 customer rows across 14 columns. The checks below verify each group against the line-item source it came from, rather than trusting the aggregation:

| Group | Columns | What has to hold |
|---|---|---|
| Keys & label | `customer_id`, `churn` | one row per customer; label binary and unchanged from the source |
| Calendar parts | `year`, `month`, `day_of_month`, `weekday`, `time_of_day` | each recomputable from `first_date`; buckets total and within domain |
| Category | `country` | every named country above the frequency floor; pooled rows all `Other` |
| Roll-ups | `n_lines`, `n_distinct_products`, `total_quantity`, `total_spend`, `avg_unit_price` | each equal to the aggregation recomputed directly from the line items |

### The split

Whatever strategy is chosen, the split has to satisfy the same properties:

- **Complete and disjoint** — every customer lands in exactly one of train / validation / test; no customer appears twice.
- **Split on the customer** — the modelling grain is the customer, so the split unit must be `customer_id`. A row-level split would be meaningless here since there is already one row each, but the property is worth asserting so it stays true if the grain ever changes.
- **Reproducible** — the same seed produces the same partition on a re-run.
- **Proportioned as intended** — 70 / 15 / 15.
- **Comparable base rates** — the churn rate in each part is close enough that hyperparameters tuned on validation transfer to test. This is the property most at risk: `04` §6 shows churn moving between 31.2% and 72.4% month to month, so a time-ordered split does *not* satisfy it while a stratified one does.

### Open decisions this notebook depends on

Two choices are still unmade, and both change what the split cells should assert:

1. **Split strategy** — stratified random on `customer_id` (holds the base rate constant, isolates the pipeline comparison) versus time-ordered (realistic for deployment, but confounded by the drift above and needing a 90-day embargo between parts).
2. **The opening cohort** — the first 90 days hold 1,581 customers churning at 36.1% against 50.0% afterwards, most likely established customers whose earlier history predates the file. Keeping them, dropping them, or flagging them changes both the row count and the base rate the split has to preserve.

`src/split_data.py` does not exist yet.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RANDOM_SEED = 42

PROCESSED = Path('..') / 'data' / 'processed'
features = pd.read_csv(PROCESSED / 'customer_features.csv', parse_dates=['first_date'])

# The line-item source, kept alongside so the roll-ups can be checked against it
# rather than taken on trust.
lines = pd.read_csv(PROCESSED / 'first_transaction_churn_clean.csv',
                    parse_dates=['invoice_date'])

print(f'Modelling table : {features.shape[0]:,} rows x {features.shape[1]} columns')
print(f'Line-item source: {lines.shape[0]:,} rows x {lines.shape[1]} columns')
print(f'Churn rate      : {features["churn"].mean():.2%}')
print()
print('Columns:')
print(features.dtypes.to_string())
features.head()

Modelling table : 5,044 rows x 14 columns
Line-item source: 125,042 rows x 10 columns
Churn rate      : 45.64%

Columns:
customer_id                     int64
churn                           int64
first_date             datetime64[ns]
year                            int64
month                           int64
day_of_month                    int64
weekday                         int64
time_of_day                    object
country                        object
n_lines                         int64
n_distinct_products             int64
total_quantity                  int64
total_spend                   float64
avg_unit_price                float64


,customer_id,churn,first_date,year,month,day_of_month,weekday,time_of_day,country,n_lines,n_distinct_products,total_quantity,total_spend,avg_unit_price
0,12347,0,2010-10-31 14:20:00,2010,10,31,6,afternoon,Other,40,40,509,611.53,1.83
1,12348,0,2010-09-27 14:59:00,2010,9,27,0,afternoon,Other,19,19,372,221.16,0.70
2,12350,1,2011-02-02 16:01:00,2011,2,2,2,afternoon,Other,16,16,196,294.40,1.58
3,12351,1,2010-11-29 15:23:00,2010,11,29,0,afternoon,Other,21,21,261,300.93,2.36
4,12352,0,2010-11-12 10:20:00,2010,11,12,4,morning,Other,6,6,77,143.75,2.07


## Item history applied to the first transactions

Applying `src/build_features.py` to the cleaned first-transaction data, so every line item carries the trading history its product had **before** that transaction's date.

### How the metrics move over time

The module produces one row per `(stock_code, date)`, each computed from data strictly earlier than that date. A product bought on 3 March and again on 6 March therefore gets two different rows — the first sees everything up to 2 March, the second everything up to 5 March. The history grows as the product accumulates trade:

| `stock_code` | `date` | `prior_transactions` | `prior_units` |
|---|---|---|---|
| 85123A | 2009-12-01 | 0 | 0 |
| 85123A | 2009-12-03 | 31 | 1,003 |
| 85123A | 2009-12-06 | 61 | 1,415 |

Same-day transactions share a cutoff: because history is aggregated to whole days before any accumulation, all 18 customers who bought `85123A` on 2009-12-08 read the same 79 prior transactions, and none of them sees the other 17.

### The six history metrics

**Volume — how much the product had sold**

- **`prior_transactions`** — transactions containing this product before today.
- **`prior_units`** — units of it sold before today.
- **`prior_median_daily_transactions`** — on a typical earlier trading day, how many transactions included it.
- **`prior_median_daily_units`** — on a typical earlier trading day, how many units moved.

**Price level — what the product normally cost**

- **`prior_avg_price`** — mean, across earlier trading days, of that day's average unit price.
- **`prior_median_daily_price`** — median of the same daily prices, so a one-off promotion does not drag the level.

Both price columns are built from **daily** averages rather than from raw line items, matching how the unit metrics work. A day on which forty customers bought counts once, exactly as a one-customer day does — otherwise busy days would define "the usual price". This matters because `04` §8a found 1,681 codes (40.3%) selling at more than one price.

### Comparing the current line against that history

Three ratio columns put the line's own values next to what the product normally does:

- **`qty_vs_median_daily_units`** = `quantity / prior_median_daily_units`. Above 1 means this single line moved more than the product's whole typical day.
- **`qty_share_of_prior_units`** = `quantity / prior_units`. What fraction of everything ever sold of this product is being bought right now.
- **`price_vs_prior_avg_price`** = `price / prior_avg_price`. Above 1 means this customer paid a premium over the product's usual level; below 1 means they caught it discounted.

All three are `NaN` on a product's first-ever appearance, where there is no history to divide by. That is roughly 4% of rows and is a genuine undefined, not a missing value to impute.

In [2]:
import sys

SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from build_features import build_item_characteristics, check_item_characteristics

# Apply the module's functions to the cleaned first-transaction data.
items = build_item_characteristics(lines)
check_item_characteristics(items, lines)

print(f'(stock_code, date) rows : {len(items):,}')
print(f'Distinct stock codes    : {items["stock_code"].nunique():,}')
print(f'Rows with no history yet: {(items["prior_transactions"] == 0).sum():,} '
      f'({(items["prior_transactions"] == 0).mean()*100:.1f}%)')

# Join the history onto every line item of every first transaction.
txn = lines.assign(date=lines['invoice_date'].dt.normalize())
txn = txn.merge(items, on=['stock_code', 'date'], how='left', validate='many_to_one')

assert len(txn) == len(lines), 'the merge changed the row count'
assert txn['prior_transactions'].notna().all(), 'a line item found no history row'

# Current line vs the product's own history. Guard the denominators: prior_units is
# 0 on a first appearance, and the medians and price mean are NaN there, so the
# ratios come out NaN rather than infinite.
txn['qty_vs_median_daily_units'] = (
    txn['quantity'] / txn['prior_median_daily_units'].replace(0, np.nan))
txn['qty_share_of_prior_units'] = (
    txn['quantity'] / txn['prior_units'].replace(0, np.nan))
txn['price_vs_prior_avg_price'] = (
    txn['price'] / txn['prior_avg_price'].replace(0, np.nan))

print(f'\nLine items with history joined : {len(txn):,}')
print(f'Ratios undefined (first appearance of the product): '
      f'{txn["qty_share_of_prior_units"].isna().sum():,} '
      f'({txn["qty_share_of_prior_units"].isna().mean()*100:.1f}%)')

(stock_code, date) rows : 100,158
Distinct stock codes    : 4,171
Rows with no history yet: 4,171 (4.2%)

Line items with history joined : 125,042
Ratios undefined (first appearance of the product): 5,659 (4.5%)


In [3]:
ITEM_HISTORY_COLUMNS = [
    'date',
    'customer_id',
    'stock_code',
    'quantity',
    'price',
    'prior_transactions',
    'prior_units',
    'prior_median_daily_transactions',
    'prior_median_daily_units',
    'prior_avg_price',
    'prior_median_daily_price',
    'qty_vs_median_daily_units',
    'qty_share_of_prior_units',
    'price_vs_prior_avg_price',
]

item_history = txn[ITEM_HISTORY_COLUMNS].sort_values(['date', 'customer_id', 'stock_code'])
print(f'{len(item_history):,} rows x {item_history.shape[1]} columns')

# The opening day is every product's first appearance, so it is all NaN by
# construction and makes a poor preview. Show a mid-period transaction instead.
mid = item_history[item_history['prior_transactions'] > 0]
example_customer = mid.iloc[len(mid) // 2]['customer_id']

print(f'\nOne complete transaction — customer {example_customer}:')
item_history[item_history['customer_id'] == example_customer].head(12)

125,042 rows x 14 columns

One complete transaction — customer 16894:


,date,customer_id,stock_code,quantity,price,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price,qty_vs_median_daily_units,qty_share_of_prior_units,price_vs_prior_avg_price
64371,2010-06-22,16894,15036,12,0.75,57,2827,1.00,24.00,0.70,0.75,0.50,0.00,1.08
64229,2010-06-22,16894,15044A,1,2.95,23,110,1.00,6.00,2.95,2.95,0.17,0.01,1.00
64228,2010-06-22,16894,15044B,1,2.95,18,124,1.00,6.00,2.93,2.95,0.17,0.01,1.01
64227,2010-06-22,16894,15044C,1,2.95,18,75,1.00,3.00,2.95,2.95,0.33,0.01,1.00
64212,2010-06-22,16894,15056BL,2,5.95,77,1196,1.00,3.00,5.85,5.95,0.67,0.00,1.02
64214,2010-06-22,16894,15056N,2,5.95,70,703,1.00,3.00,5.87,5.95,0.67,0.00,1.01
64213,2010-06-22,16894,15056P,2,5.95,23,165,1.00,3.00,5.90,5.95,0.67,0.01,1.01
64329,2010-06-22,16894,17084R,72,0.21,9,833,1.00,72.00,0.21,0.21,1.00,0.09,1.00
64397,2010-06-22,16894,17109D,3,1.69,8,402,1.00,6.00,1.36,1.69,0.50,0.01,1.24
64211,2010-06-22,16894,20679,2,5.95,52,737,1.00,6.00,5.85,5.95,0.33,0.00,1.02


In [4]:

# mid['customer_id'].unique()
mid[mid['customer_id'] == 12437]
# mid

,date,customer_id,stock_code,quantity,price,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price,qty_vs_median_daily_units,qty_share_of_prior_units,price_vs_prior_avg_price
3670,2009-12-02,12437,20724,10,0.85,5,110,5.00,110.00,0.85,0.85,0.09,0.09,1.00
3672,2009-12-02,12437,20749,4,7.95,2,4,2.00,4.00,7.95,7.95,1.00,1.00,1.00
3673,2009-12-02,12437,20750,4,7.95,4,29,4.00,29.00,7.15,7.15,0.14,0.14,1.11
3680,2009-12-02,12437,20977,16,1.25,1,4,1.00,4.00,1.25,1.25,4.00,4.00,1.00
3681,2009-12-02,12437,20979,16,1.25,3,36,3.00,36.00,1.25,1.25,0.44,0.44,1.00
3683,2009-12-02,12437,20981,12,0.85,1,1,1.00,1.00,0.85,0.85,12.00,12.00,1.00
3682,2009-12-02,12437,20983,12,0.85,3,26,3.00,26.00,0.85,0.85,0.46,0.46,1.00
3669,2009-12-02,12437,21429,16,1.65,2,12,2.00,12.00,1.65,1.65,1.33,1.33,1.00
3674,2009-12-02,12437,21432,8,5.95,1,8,1.00,8.00,5.95,5.95,1.00,1.00,1.00
3685,2009-12-02,12437,21754,6,5.95,6,51,6.00,51.00,5.87,5.87,0.12,0.12,1.01
